# 🎬 Montage automatique Locapay

Chargez une **vidéo brute face caméra** : le notebook la **transcrit**, **coupe les blancs et les hésitations**, ajoute les **sous-titres animés mot par mot** (italique fin + mot clé en gras capitales), les **recadrages**, les **écrans de couleur Locapay**, le **logo**, des **effets sonores légers** et, si vous voulez, une **musique de fond** qui baisse quand on parle. Vous récupérez un MP4 vertical 1080×1920 prêt à publier.

**Mode d'emploi**
1. Vérifiez que le GPU est activé : menu **Exécution › Modifier le type d'exécution › T4 GPU** (gratuit).
2. Menu **Exécution › Tout exécuter**.
3. À l'étape 3, cliquez sur **Choisir des fichiers** et sélectionnez votre vidéo (et votre musique si vous l'avez cochée).
4. Le téléchargement de la vidéo montée démarre tout seul à la fin.

💡 Si un mot est mal transcrit (ex. « Locapé »), corrigez-le à l'étape 5 puis relancez seulement les étapes 5 et 6 (bouton ▶ à gauche de chaque étape).

In [ ]:
#@title 1️⃣ Installation (1 à 2 minutes)
import os, subprocess, base64, gzip, sys
os.makedirs("/content/montage/polices", exist_ok=True)
subprocess.run("pip -q install -U openai-whisper", shell=True, check=True)
P = "/content/montage/polices"
for nom, url in [("serif.ttf", "https://raw.githubusercontent.com/google/fonts/main/ofl/playfairdisplay/PlayfairDisplay-Italic%5Bwght%5D.ttf"),
                 ("gras.ttf", "https://raw.githubusercontent.com/google/fonts/main/ofl/montserrat/Montserrat%5Bwght%5D.ttf")]:
    subprocess.run(["wget", "-q", "-O", f"{P}/{nom}", url])
open("/content/montage/montage_auto.py", "wb").write(gzip.decompress(base64.b64decode("H4sIAAAAAAAC/8V93Y4bSbLefT9FnRovumpElki2ukeilmNrZjSSbM2M0K2zY5hLE0WySJa6fjhVxWa3Gg3sG/j6+GphGD47e4DjG8Pwwfpq+032BfwK/r7IzPoh2VJrdww3JLIqfyIjIyMjIyMik7ZtH3yXJoW/CCx/XaSxX4Q/rQPrz/9k5cVVFFiv06m/8q+sP//poH2fv4PfhLPbn1Nrkq2LwJr708Ca+vHtz5lvWe0vLavI/CSfZuGqCNPEcn5chvkqyFyViZrTdL0KrFmQW5PIT6b5AdPzdJ23i7DIkOwnIcDlVpwW1srP5NsJCz8SvOdhYj2QtGl0+7MVJNYi83OgsGKRIHcFXhZM/VmGPgNcVARZQoAPLNRYBJn1Pk1jawqihMla4QW0iDbQQvo6CtYZH2M/Q5MCMEoXqSoazOdBkQPjJCW2CmSu8uJ1Lkii7jxNZpaTChWSIIqCljVbT8/DZKEw/O7NI+siyIpw6kdWt/O4c9l90utYq+z2j4V1+3trtZ5EIXB1HmxR1CsuCysoLC/PCtc7sDG+YbxKs8J6B5xaQLpYtqw0b4EILStfroswwvd6ssrSaZAjfeNfBAfzDDSY+YU/jfw8R0c0kDKpBVIH0ezAQE/W8erKAqmTlar85tVrU+lVDFK31Nc3mb/Rj9+GpL15Ab0PDn5sWS9b1rdvzqyB9Lplsdst66hzcHaKtEePO53OwcHBZ9bgb/izMoxKxOE/+Ddlhw7k0zoNVE7/wMKfHu1xt4/pkAEB+7Pe897Xx13bqv19Bl4N1iBBWJTzxTE884A86Tag9SponWdf95492oLGgTeApOIqzUOO7pgTYawmQt+aR6lfAErHOzlRFTt4W2Iag2x4mvi51E4m40BwGWsE+laYsGJPshXHjjXHlqgpzrVraJVJ6dqy/fV0ndgC4CKN1nEw1txdx6v7WArk69UqC+MgGy8DdMRnV9DQJE0jFHubYUpIA8EarLlcxy2gvvQ8T/XdX+cA7l/WAT86tpp/n0F6tWdBdPv7lgWpJbLDCvKCNF9BEDi5GgLM2UUw9i/8pGgg2qnnrrbI2z2W3Nk6C4LxCqCb6PS8R3dgsk4qUSPYhEkOUXj7s8CjnKmgdE0zAQQVhAKqNEgkeZQz4yDBQIbJbq5CcM6sCqoagnk4XUJejAmgHGJbM1cUToNxHmThfH8WRWge1PIODmbB3FoGl71sMXGWrposS3Kfh0LhynG9SD3Yn9mK8llQrLPEKtarKHDAf85yGPZDTI/eCOx64kIkZlYIClkOJnyvZT1y3b99pjeEo2A9xuKQj2fBah2C6YNFHCRF7pgH3RWWQW+GI0U8YIYCxM2UU8VM5kZneYugcOxNms1yu4XaroXM4agqzD8OC2jjbIZS0h65hmaNYuHcKpoVDWKev1oFycy5tjd238Jkt3NbjziB5oWfFYCK9KCejipIvWmMBsHp0cQjZVODYI4mB1aROLq0nO/8qfVshQG0zkJwRpqgkbmfQ4q3N2ohV2VeJUUQWQ+tN1+jQApk/bAq8HUa+RPXszWP6TVCfUXhxFMrEqZZAdLGB5oW5t3Lr9Be7LjgC8v+xs82YWJDKZhVJWIfvJ4EuoifxSePVIlmGx4mymwMnKYOezfWCNpuRXVNJWbbBpG7gShKfAiOKnEPUB+AocipYGR+mAfW6RqqShw8z7I0c+xnFMx6NKlpNNUDCCCIF6pGDjplBmVnFCHhm8OG8eJs/O6Hb56/fn42/u71vwcXX9tAaZKCyUih9jSN43USFlcPdaV2RHnavjhqq3Lk1Bit311Bstskd2ub9e04mIXr+O66Kl9Vxlqlm74HcqxxU3bt22dnb5+fNnv3gX6oh1YDPfW0hUT5fKMnnBmYLHAuwlmQQjdLsW4EA9u0gpVmscb7PMPLBfSByTryUX5g29XEfNsYX61PW9DhptYyzVLqN1h8GtqyZo7pMg3zEHILyw3asvS8qaamLje4QzYYNjbFBmqiVCyrp3ZtcpVZ0AewBMy3sCdPtqxrRYcb9y+/+2/o+Dxa58sBF7lKQApSFVjPYDYpabmCnjtOoXHMx1mwSgc1zhUprdowNHc1sUGqgaL6Dv/t+6P8HnPuYVLFq1yQbGGOQVnzozE06nhVDGoDx+Xg+zS5H3CI15nS+vAPKslFKMpfcFkMvvWjvE4NJRr2r2wZVgT9DPG/Z8y0UKqtaFTfm8LMDKVmr+9ItI+NZVOk3GtYYwxrvQmnOSn3D9wMlJliTnD7CriY6CtsPcfF1QqJwOyxXTUASmDnMib37OGZbRb4fze+v9DYNpoZXmvNo28e8bTx+ECBJVoBE+SJykGiCuD7ptJinFxq5EpzcUcqK9cKTm44SLFDkWbTZT2hPs0xLtQWp+uZb4s2w8IeXz30ASp4CD0g4jIdoMcouFqXK6PUHag0WbrVaAuCpXhUIrUhZys5XFs5FX/af/nPv/8//+s/WdYbXzbyL978vdUn3Ns/APChyPFDiwtxmHN5XIE/sZVIZDcdQ9/Oufv+eRrmrmfZWwNsP5sW4UXwnhq/Avz8EmXXMhO+tMDM4ZzbdbREvrRmh0E9/+0j1vJqjKr7OzALzEF9on29ZH/JAlD5yy4Y0Q/TjZlqsNpg85VZ16DnDaed3gQBrpGc0A5nYyleziw9ofCldcW/abrsnwr3mEPWfNU9GTglK5CP3DvAfWw+tbinnqR5UJ9edyr5DZF5T01/V8cv9fv/7+r9y+dnr94+e/vqh+/PRK/BPpszZRms+YU39ap0lmUsX3EsieoT23H50hWDJZSYs7c/vOFOhnSwbfAfNIiIFi/OgXWibHiYZ+DQGacQVMoY6qo1S7EzByWTEFa5zHqHGbG2wsiiEczCbEgwclB28BFGuaTmBzEKBWiK8xByJLSiNf5zAeNHbl3R0EdLI9rB/4JPuXX7jz6asKbWTyhNu9tPa/xDbaAAfG7/hTj5tKb5+F5fHnCHnjMvh5TFfh0vcUxAfyxoKQxp3vIvYLugevXeYkn0c4VUERZFdvsHfgb8SFfYPQNBKl2ijOU0BnEy0oZzMOPbdAkoFzQPxuh4gf9oHVSyCh/FLTbNhml+S1LABVn4maQkENJDLnagDewFgAE7CVImYUBQ6+KgIA35FKjPHBt2MG8KYiRo5iINYaOQLwwcnvAVTkN2Fm2gm2EGJQAWrbnMyumhUOcQI/XTWn9ycA7eHYIwieTODjH0/AAzHKIJ9D0OMTQKIEwuFkwrpJFYWCy1mllkduYFyRQGKNqA3wGbWfDOty6wweE0PHi3hi5hxbd/RG9jSD60zlVBuo+qpLSM7CpYF/y4BFnVB7AK+T8LwKVevorCwoFh4dtnr76CYvHXMDD5RXMM2W5mxnXPcFeMdwB4CZn93cfZfZeTp5/EIAeGQa44ZcBcoBnnT1HnUOHYkoPqI1VyZYNgas8CE2HsFFrYaSGTBR7Mx05mD//jbze3v7/9L7f/9fbn2z/c/vH2n27/+fa/3/6P2/95+y+3f7r937f/ePiX3/1De0QRgv8F1p5NkAG4Bw09gqcAa/Tv/oG5h7ZpELJQDPsZTF4ZbGliOuQGJqemV6ZU26FaIlbhw4hW1NufB9qa+pTsBSsf+rjOBrXn/NBsegoqJRSTN+WysJIZEKquCkHQ2ae/TUZ2AwlKNbuueEDG2wObFQVCU9z7sHKiGcnRUFEYprCddcI3C8jueiHIDmVUylLuiLbfxqITcuQKVXrHoiCLBN8hJJrLYUzUmV3VOUcJaS7G6mb2E7qJcxbfakNZEZMpIaOfNHFAnSH9Wt7f/eunfSglowf/Ct3ehmf0lFk4hc4P3WKg+no+gsnQEZDDzkgsQwJe6ZC2W0FAd8xSGjdWRQpHvQWHghBsbWvZ3yaD25btvYMCqPvcJM0vYKOkRVn8ScoPJ5hBocEArsEbYnDWHC8G3pZ12i/dFFv2SkUtt4mi7PegIAD1L/kApcZqW92gfTQy3HHq7bfRH9QVEzYQ74Kus4PmOrttu8JxKFZTPkZ1ZmzylbJk/caP1g07lkimNJJVocA0u/05UAIXsvpCHI1Gab6DX3W/ocD6l456aRkaPIBpv/MYA6jM9nBewiSfKKGvm3UgbpVHMio9kiBi6ZNwtSGftRpTJ2wpbAL4xYLML4I6X2kqqGqyuzGDIknDdnc0tDFVuOlD8q8HGJ/KDdKYInsrDHS3n9az2T50UT0jwmqecObsAVqpobNgIoD7Gk8IaNNW3zI0VfCx9QxHN9WIvCcNBFzVwvuh7ZdDghFoMcW0QSKcejUvjVuvN1H1MBX1THhf7/aDqib1C43Femtk9qGEwWA5DoXGDjRHiqLcxPBKr0mmRgHVnXpSS2PsPq1KqkGwHgyYp14+MA6oZkbhfZPTkFN1CLz2vslrUsxt7lzEtVK22myIaRi4oc2K0pn3YX1aUATllrPPbd7S84PyA2ZlqmOyWkK1ELkmmjAMjAvtAr2ayuo67HLgTz36wLDmeZ1H5k2RWosm1fBHx8+3Bnrgnlrh7NLaR17WlRUK0n4ROF36CRMHpd2tdTWWDWRLVh9FmNnl8JwSc0TZUaaMRtsLdTmNfciIU6/pL7Q+R7++ODZMNpGCusaXzNpd3aeyIDsKISXIHugqLrwrvZ0KQi7DMkNoGIBA/iN98HhwZzmFkF5+S6Jfg6J9KirIhLcR3xwgPMooDkPrV0JDeTOWIjCjQ9XGbXKkwHQVxZRTtVOpV6IhsUDNZjO0U2HE4inKY76sDMlWMs7an56tiGhR59VFRn4UIV4LG5ECOotkyYSpWp8qslFxS7fJxMolXIFMzvImWyRTGGu4zVjHDoklJd+N1HIp7b5jo6jrMgIlSJx9mhCadkruqriBfHPUod5JHTKgLttUrahZqeI7IEs1KVODSAS+RISF7M0F6S+tnrzFpVz4u4GGpt7vgqcX3XZFls5oZDgdHuk9iqweGsOTxOZpOUwH1YTIthaw7THZA0fzht6pY7nAxnad6JKyc8PbKgCfKO8I7Gdc7P/8pxZNadiH6ScA+POfQFQX+wps4Fb0iXMfWAIKuV6V7LQIK2FDAmvMXAqSGgEWLSuhwNLZw0UIrqxewBPdqv8w2lEpEXAuaSnSRClgJPVC+E/zFlrXu9xtaqtKxYeYZgfcPhiNQmbp0JMhuVSDvp9TytqmWF7xiwFIaL8WBt/il+TSgzUWMTIMFFh4q3QlG1RmvWiuFQvMaa4HjVltBqICulGqrVoEq8USNSvE33NREW1qkwNl07GqADQYGv1UZn3hSS4LIyG2xrXqOthYpaHHDWaRbU03eFItY0BMqUCA22J7olgEWkc5KbWOatKA2NJB4Rthlo0aU9gQz5UMOm8py2ZFpU2+R1wYbV/XNyq+mAI5pDsNiESpzV9Z+6koCVLusCtdVxhKV2n5LYu/qHRQrV1uILypK+IRn0oRpRE0RGSbDehcm2jCtiXgCW+Ed9NQ8ffE8jkxjP4I4JoF1b6inIQslRWaGolRW+kFKpxTbye2yq123CVJXggdTiqGK1C/YIjWC+EWdgiz/kWpx9e4h7Th4IUVXyqZ0msZ2G2r57JBCYv6NYGGBgzy1Js0wcye16kBDyeR6JPJyoswp1Y0wnDGtchKaCFxVU8oamR8hmxkVHet5E0RTSynsqIJ7C2z+JSrdQecOoU65IAEbbw2GQ3hEJnaTdW6LFSgKQS2A3+SOyGXQSUBKw0CuLg7OhlZVIHcXXSQ6p83Uic0Y6rppSphhgRXcHDEkxnMo1CG0LhTJ+mDOrlFK2Ofpq67pSaE5ZrENmqzU7GNdNgEVJ16Zdwhpuhuas/dUmtbikrV7M3hjAtmjpBkS3oqdNX8oIJlEICK+yvEYh3ULR/lvFPmB5F5fJVvvIvShnf5xrueMkh5wflI9Ywzkt9Nx41dpAhDMXmycyuD10x8RJ5NezDViOqI6aK70VAYhxqDUUNJUxqiMnGckOcLSFWlPzZJoXupdU1E0jbVS50tIulvD0SjqUkFoKmoOgd2x5hyC/slif+98LNQzC4MzYU9UHe4yK5qGzkMWBkuC5fcOqA7cQtUxVyNulIfC34xLlsaT67GiR/Do2dSamaBy2mwKuDH5BdytvReRsxu0XF+8OF6utge/KW/Ffpq/EOW49beEdrBH6gpgugwrHyZsp/O6YmbTNJLVUrvYHzKzCO1xUC4MSMTMwxuxHBDSB7ks3abWwuToEygMSa9obGXBBvHfg1tyCEMTEUtqspAZo8fjlRyPbbvOGy6TRTVAxt4hJqmC+j2YM7Y6SgadI81PNOqfMNGyr21UwuR9l6Ib8dPvorWmfPELDFxE9HTF189q+HKNUr/c6FLh7GHsAg/Wi19jbDiLlFO7gllX+cB4K6u39Ftp3d8jADP+kfZI08QHDOGgyHPgYCvT8YwbqnRy1RTMohgGBW8/UZmlpb0MtfGdDSPx4jvjND+aU0WMhubmRhueefz6dLdKxoS6di0tqotKWHS3GN8kRdchjnCMKbUpKZLrOX1uA2w9Jhi3TRiPzyT0MWHr7Gy+NnVQ86B/OEZDSfizPejhy+CNFuEvvWK5xmmXlHM7Q/FktgP13n2MMdmLXg4F2hmSj2k/+xi/fAbfP1mfcbY3nYN6Daek0/E8xnERWR9lUazXw5HCPJ2CbEeshPNPawf+mGiHzKTkhF32EdrUcyt6l2FLrfMcJiHSRP8FJFv2udTsg4d/Ipr5CBMKZ2avhCnmdl0h2j1uGqiv62dqDM2DG+Y2Lv6SVWRbo9BuXKUlDBLh/38Ek4MEs8uqdJUQIKo0dr0k1vLa619JyE2mknLFvPtFrfNmZ/WyN3Q9dxsgDJBzPAiMD5+BRnlQAc5bekY9pqH8A2ieNo4VjNdK5+vOVzBXYQfyekcmD/othVzgBx0gLcVtjjldFbRtfRM+pl/+89FFS8pW06O+IuaPoKZtKUic7NXaWBQoKlhNEnFfQ7Za2HMmC0+coOzo+Ce74egVOmFdkwM9Y5MgklaDE3JjflJ9n2bfMtstX/0lBOfW8mPgRv2t+2j1f5v6FQu45b3tK8cgHQJo965tjd4EDd0DovxscxhM7tgVTjBvdA65ya7vwe1ilaRVIgk1kw63FJ7SuVUIMmjZnXux7aG2HCDQCUkBX6XoiykWH5cMj+kMdt2ePzpMZUH5+Rxy/qix6djPJ3I0yM8HffcPbY0tZQqgtSUqHIqSOxSXtSbo2zQYkjt8wxOXNmK3P1QPKN0gVKSJcu+7jcOBvkKaHHr3PF6j6F4le3sLR5ttOU05D65V7fNxDntpYT3uTKISUp7231emygAhr3Aj2KHf3zSv7NLu7vCcpDNFo6NYfu9aYkxQJA72kKuRdy2doFwn2RyYOhRc1e81L3MTBczs81XnZRqups6daejV5quL1GQq9/ugTFWQlMPtx0HgrbpT7QUxMv2d8lEY6XzI7WvjbsLzAB0SkVtQ01y6RrmQEt3cCwFZY3ALbTUViCu6PKPlngFIJMIrtw/1Jd0F2w2ijt2SlwxF7A0WT/gfVNzty5VdgRKKZ53Z75ibJnIT2CUewxjyheYzyedPRP2o5PV5jqoVQw1Ic2s++CMa8y2k3K27ZLtl5ln95xju/MrWmqz2t6Z1Ch62RJObzAgzozKN5Pu4O9PY8dfhBX3syE4RqCL6UWetOYyDvw8cC41d1xqIxSJckljXMcVL2ljL0TrmcOPS9f6/HPrSEOaphjjbMxz0gHtrBc+5rZ0AuiDCroJGko4WJ3tDbraxVM3E5RW2DzmU3HZPe6JjblLwR0oryKRFFcu+aPrnZTWT1Sh8f7JkycV/AS0T8xYd5UJFXu+TTiDXPocVcjczTyELC6Whc6sHSzpkXwxAsfy8H3gOAqyq88Se1+9ev3q++fPTqsaMhpVY23gopjnqZIItbbahKQY6oGg0X1E3iehA41DU1gYbEzXffS8u5vdg/1udeXIXrxXbcb5gjkOlZMHwRH3YrsQ32xXWx0v5LiucwEkfFcjoIZ1Z7cMWNAMWPrSVSS8cn+JU5Rbh9qd/CoplojIYaQ8Jug4SxbKiIw1Y5bGHtjQX0cF050vjD1nPJuU/G04uEPGdS5pNu2U5cSzNp4AVODgGoEQYm4ODpuDK34adL2ehkFbPAWSFGGQxVThsAjSWFmyTa0EuSq4DdnvgyzNHW3oQgSiHJQWGztmGeID5dBsBxj9dNAInVBW98RtGuNoPOKRei+X2YqHVYik+ZSepIfWWY0Fl+AuNkZ0mduW1tto8XPBoCwo6IAn58hgpadSUCcwU3pDEIOq4k5sWz7n8ag0zZcMlhl0PJhaVuEUD9Qm551B71jIMzjGkXrQCMcX/PWg3T2qkZcchMpoFh15ahXbngLIJFJXzYnkQmVvlgGOtInNNZyCHQvQAU8io3oeGVTmUkFjuTwxU3JL+ZHLzrDGBuQkHlxB9CCOGNDF5EeOnD6VPrgUSMlFnRA5oQMbGuxxYoNShQsZTMJPWJrMqLrsClmQhI7WSEevYkmTR1s0gQzsbhHFN/who35Q8scJvR1POmB0lgouV06bBAGvdR9XnUUWGainSgkL4Xu6jrkozxVMdxfE0bFrRNNWTqdrjGo1emyTY4sMjMmoCADtCMpCdQbY9B2D9PGu51qf2NMvMnGx3RdHfFe6Kzxwj+dziC5myZbN3VnS4fSYN50ew+5R9zHZvPvki2N+906OvsDdEU+OuyPX0PrzgbWPXEf3JFa7+6RBqUWQgNuzMSjmBBd4kVMjUAno36BRkIb5LQo6kknLs2so+bopm55ap9keaYXGcjnAoeY13CfVJKc3Jl3pJDzR/6JHUCea15t6xAvctTGdK01KVj2pibs8RKxCOidXExFEHcej2qGWvOwdyItium91q0dOlaCpbgEoWSWE0z/vI3qLUDqVUm5c3dhe6ugYFK0pAiufRHXqQqktkk4+AmCS6xnC49Z0cKntbbvrKs8m4s247dUEVZkl2VX9srHXw7wfSPAe8e1L7kgxcv4ToxG8koWBlyujeN8qD1QVxaR6Xk0jhNKyJG4oOXeGr2HOynhXgjvsk9CKi4TISG13y9DxTQgNh7e4eDzDXTqH7M0EUbkIgdnUIh/oD0q07pE7PeAsSTnOb4mmVCXNM19zR31Ucb4wg+YhmfCM5lTVjnpfnHzhejj+Sd8OOgBsccGDV6STKxxXcH4JjSSDar5W6oI2Ifhz3nER6NNpmm/Vql/dbuNl68SBs3aO94kcHWxfyIkmxhzLK6yKiDMeI4458GNakeyLfkflLNPNmEckQng4P2jxVpUHQsCW0ir7Km2cA7kxz2QPslQFWZschH7nKjH4MPR2Oic+vM6Hx8LZW4w/jjdAdAVj9BiqpT6pKgfg5IgtmGiW6ghAqjisLWf/cgfJ7tA2HWZQSt1tNsPel/3g/hfPqjfGQZoW5XSV43MzipGZ2rKWPR3jQGWx/zoMHpAwdJBzEvlsyw0jLcipRznnlgOFsoIR6qoQPynTqLIoDEhRtHt946p3Q1y4sliyU+5VKOeR6UI+dB93KBCe1MSUJgQk7Ka+RjBZLwLKdS0j0eA+4xdVqH8KEzZYDVEuVHiKAdpRPb/H+XHDJtP8YrAadD6ZU8rTJEYrx31SOFCJY53l+Z/YiHx4/bFR5tTvQpek4AOxJHcWXuBUqUPr0tFJh3/IjHczT3RWvpslIBvktK+X/U5vdtO/jvV3Lt+ta2zmO0ezG3NBTSC3KxBzHUTQWI5FTG5JSJCJx9BmuIdrYK+LefuxEpnzpten2wi/WFThCCYAYrT3rE8zomGhQ0igT+xkmOAdHtLYPQ9kLP7k4knzBRt6f08gsbrMLNiKBRDRjfPE1+c3v02uayPsuzeIlfrSqqdNXBY6tA7VWZjN8HBzOKqbwg5pCjscsdhvExuDec51r1teL5PQq21uZ5CO1s6xgAYIWQl5rl9dIzVQZ4AxRRbYcubges2pGe0ykWTrgdSWMhBY+38dvUA1vEFbA9TwDak9NQOy9oXgKEe5Vuh0yLGCpqJYRi39qgJempHrjTCVg+oSJxVwOlAvi0ztQMvoNzmSU7+zSWLJttzOzRI1TS1alD594e+tguhawivGtLMfIyUVooU3RbC+Ey3KkIqaptVEelFaXo64eazZbI5kq4MC2prykM+ygrhuaZ95/ez7r//DD2c16NyGf4EL7mhD1OVVVQ2GccMnSj171Os0sTLUq2FlTPMaKYD/NJwE7kx4imBk+FR0nm+Gu3b2o20dedVqQi1VhbsT634TVw2T8WX6tQHNZ+CmDlbcE6jIo72cjIxsz6aKVVT4odiPjUSRPRRPVBhJ0qo5HN07ZdauY5IObPLkyOBE3WidBTpSUmHRMkd2s2DGqYozWi2ZejwpGx2iG0s57IoQ6ml6wQsK/sSoapiP5J4zFFda74bXEXLjv1+hK/eVHJAfMYas8JLfS5V3hssDz17qJUlMMdMgjJycY88Nney2etouVstfNvPVlXbc5cztOS4xuMZthDetHA7jYHB99iNWm7OXN/15RHVtEk7X+G9Xc7Z2eVs58AAGQWi3gp8GFMSISChgxuqc9HO/0Cs63x/3F34c+zDQPHncghE3zSY+D5wFgywfcLPfn+SDtjxk8UCOI01iSegpBDg46r6emq7xRiQAtY14FSxEuwDnRdjhRU2dIzQqgugj1B0u5nwULQKRPpJ3D83Dbq/Cy/E8LqTeYtJ7JODJhEqzGNSxe/XmuZavyl8ywDByT0M31pG5Dq+QTS/CsbGT5T7HDuOFie2tAjaU90TxpJzr6W/FzRPSUMAwsBzfzYVysp6LRVGIaLQgKMXQbPa4TfRERyUXwFSJ+0SY1pF4UNckNNB6PhTRHSxrMECEUSnoeWkPkJkzgAdinSFcakKA4ZGuxpAjoTI7jYPGWrOqwVVkLeLVmFY9tTZjk2p7wgZevHpkG+mDYw8Sk/7XMN2VvYfF9rLQfTRe6srQZn68ubx+eSNw2Arv2sMcdg2Tgxf5Me2L0o0w38veieJROviD4l5tQY5d8YIjBSqTHnQfb+N+tb7AarWi846UVFMgTPbPgGTSVKf19p6oY4l+19hpJTWL9KTG6LITSiDMUKu+y0ppjhX1ZCvORCbGu9rJA71y6Qv1sC69U0cVJEQW2mVaaCvqLlu+22XoldaAeOSpafchojpQFx61VIfepjsHn+KFOE/qU7skkVgbhD7bfvZM7EL16D2q4C0Vw6dXTOWp4V2M6PoE7CEq5c58FFi1tVA2mcDqzqgb5ZvZH1XJ+20ZvKGgMqiwS6u0hES694mfgqFlqsLOylhlGklF10G0c2v7nNyuX/K9Iryc5BuJ8cnr6OOttEWxAVd8eZ2j3a5tuIfiJV1o871aft/vej47Ne/l2Y86Jn66scRXS0/m2Y+SALa+apR9KY7jY2Yua6VfSoJ7N6UxHqXqp0lsHIFf/z3+Y+zTy4FzCal3hf+XcgJhwxc+QAvcVol3rM0f3+NVkV01HazTLTnM6GL7neIqCMP4aJVvuvIV3x2s8hFnr+ITxgvQzA0/xs4aVqn1dR7n9D/16rfNShKgfTmoFFkmlfPjjsmwD7+qURUUwfHt8dCJQtfAV0jDx/GBII17TLZp944ZZrqv9g+17n9KN6RySwcklAl6d6HDE5yXzUy1DdG5H15slGQUya3IUaeGXns9WVT0bl57huv8bDdMr/WDEtXmWp0JhAGMSyVm4bFcntrZPuyq7h+zlEC2rpObh9fJ5AZoXneVi4t9Ysqv3Duu+EOzage/w29sHY6f3WZVeYfLWqLDWre6Po1wtRc962X6xue1NU9Lde4c+plTPwfW+MMxzlmYKsgX9UvBlE7fuCCbh2D17dcVmpM0U9GD+uDNdVFbdMa1RUfCdlRoIQ9J37gNS5LkKFhNEqQ71iO9DO9ah9IPczL7V4a2GL8H/Do1zpiqM2HDxcd2ihKPujOOu7Gto23Qmkp8axywWkC/l+OPJq5qaFrRUaFwfehtvZxZjkRIPMQJi9GwfzLatcpNo49Tcm9j+Z4T2J9EWTrjmmQVznrAfXpFdjWxISEqZ12rPBMFga2DXeG+a2rhSACLX9jGLVjzQda8j0jQLG/ujVDjqI5xaTsVguAoy/wJhOvuofjzj94tMVO3Lagl7311eqoEbmgyt4edvj/yYVeOB9fvh4f+4ajvHc2xo8bLRL+0EHNUrIp88ObtWfvs7bPTt3ho2ful5Nz25/4sGBQD3Hk+kz1xy6RggOFaGVzr45oztSAjukGalMJHQx+mz1FtuVdkqGFcL8Dtof1U3wUkXaPuhhR+6mQFwJRnYNEcZsFkCut9Mrgm0yr63fQvBqDGoHsXvooJiHPvuIZz77hBCxAAEWgD8BMMJgzzuJahV5SM0vWMoRKDVwxh6L99M2j3+q9Pnw26aBWSDD4+sWmc3mzD1P4G7RccR/4VsMsHvM8V9gFfbqsa9IYXo+HF+ehps/Kwy0H+MAT1iwGMSTke5oa8cvuYkjxb1gi+gJdLkWx+aaCaWrrqA6mrHWpRKlPQbndL+4auWBMUeoieDnsla3b6dSLuZcct6n2kt9vMO7d1/69PveZvJ9y0tvj55C72ULtDOAi8ijvwPIw7zfHQY4JkDhV9ckAyTBgyxpV/UCzxBe5Rk6cv1qjBSd8v6HPGCbV+Bn8oaDA4Ou4M472wwQX5CHl+HF4C7xV7ftRXMTrQyEFP47cazMMMYeJIjqGqZAP5RsOPnwx99K9khIaWp4xnT4m+n6yjCPEk509Vo/UWe39di6WZw28KWFEIvPiRb+tfsNhx3t3DsvG55kuxcchJPonViwLeUC7GtdgXFtXI7NcF7Xah7BoVV0oLvjFuwB+vDBo+031/KtkT9YrfTznX9gd/JB4wjGulkv11HVOzSWwa9Rdf0BCvIyIdjScTS31e3Nm3OL0QWypLP6BFRd1vavxBe1CuVCiNgG6aq13/7qOvcJ7ggBsM0E6xc8j1hzO5dOyOI64mKEgQ0u4szBrOW7mbjR+65UW5xpa60kHzdMnc/mad8Rozfdqob11LhUMZXS6C3fkN7NHOtbHGDA/FgkLPWnnQemcWXjPaSh2Gl+suFpSTl8ND0cQOG0oawOy6Fly7WvMVgN2731QYvDmYo84eQVu/NIqgeGc21R15u5dKXBpVq3aIB156rqu4v1phZTAa2aoL+/wR5ra/g+bpnWrdtqxrdB4bcRD0xOvNb/ALTJXvUu71ublGyzfNHyex4bjU67gApPM75ClSHpIejyVUaIzLnHDl2Fhr//ria1w7Bo0gV64Mn2Yvk+I9yxZrxlK94VumI6Z8RJfNZmNf5zm2sn3yrqbtHMV39h312m2hKK8JiFYD+9+e/fC9uksQY6bv/+ePWyFuKql+HsD9ADS1Eu3FpE2RQAOtju/l7xDcBUcOYjaL7oXIA5r7IJKCiBgkzVg+13TjDzMYn7FT92oOfDFXtBo/Z4NEfSC0cRwUyXxo3AJdBsQ4Yrv2PWESV92/KWXUuajqRxt8r+aYkrvOBh+6tlHHh5haPIxcu9R7R54Y5MRrriuVfvMW7wTd8pcDS/VU7ZAbsQ/GeSzKGw2jBoTLezRlUwG5W8fJ/uHf2VVL7sH/BRduR38sbgAA")))
open("/content/montage/logo_locapay.png", "wb").write(base64.b64decode("..."))
sys.path.insert(0, "/content/montage")
import importlib, montage_auto as M; importlib.reload(M)
import torch
print("✅ Installation terminée —", "GPU actif 🚀" if torch.cuda.is_available() else "⚠️ pas de GPU (ça marchera, mais plus lentement)")

In [ ]:
#@title 2️⃣ Réglages
#@markdown **Couleurs des écrans de couleur et de l'écran de fin**
couleur_1 = "#2E2C51" #@param {type:"string"}
couleur_2 = "#0AC2A4" #@param {type:"string"}
ecrans_de_couleur = 2 #@param {type:"slider", min:0, max:4, step:1}
#@markdown **Sous-titres et découpe**
position_sous_titres = 0.66 #@param {type:"slider", min:0.40, max:0.85, step:0.01}
couper_les_hesitations = True #@param {type:"boolean"}
pause_max_conservee = 0.45 #@param {type:"slider", min:0.20, max:1.00, step:0.05}
#@markdown **Son**
effets_sonores = "léger" #@param ["léger", "aucun"]
ajouter_une_musique = False #@param {type:"boolean"}
volume_musique = 0.18 #@param {type:"slider", min:0.05, max:0.50, step:0.01}
#@markdown **Logo** (cochez « autre_logo » pour charger un PNG à la place du logo Locapay)
logo = True #@param {type:"boolean"}
autre_logo = False #@param {type:"boolean"}
#@markdown **Transcription** — mots rares à bien reconnaître, séparés par des virgules
vocabulaire = "Locapay, démarcheurs, loyer, propriétaire, locataire, Cotonou, Bénin" #@param {type:"string"}
modele = "turbo" #@param ["turbo", "large-v3", "medium", "small"]
#@markdown **Vidéo déjà sur Google Drive ?** Indiquez son chemin (ex. `MyDrive/Videos/rush.mp4`), sinon laissez vide.
video_sur_drive = "" #@param {type:"string"}

R = M.Reglages(couleur_1=couleur_1, couleur_2=couleur_2, nb_ecrans_couleur=ecrans_de_couleur,
               position_sous_titres=position_sous_titres, supprimer_hesitations=couper_les_hesitations,
               pause_max=pause_max_conservee, effets_sonores=effets_sonores, volume_musique=volume_musique,
               logo_en_coin=logo, fichier_logo="/content/montage/logo_locapay.png" if logo else "",
               police_serif="/content/montage/polices/serif.ttf", police_grasse="/content/montage/polices/gras.ttf")
print("✅ Réglages enregistrés")

In [ ]:
#@title 3️⃣ Charger la vidéo (et la musique)
from google.colab import files
import shutil
os.makedirs("/content/montage/entree", exist_ok=True)
EXT_V = (".mp4", ".mov", ".m4v", ".avi", ".mkv", ".webm", ".3gp")
if video_sur_drive.strip():
    from google.colab import drive
    drive.mount("/content/drive")
    VIDEO = os.path.join("/content/drive", video_sur_drive.strip().lstrip("/"))
    assert os.path.exists(VIDEO), f"Fichier introuvable : {VIDEO}"
else:
    print("👉 Sélectionnez la vidéo brute :")
    up = files.upload()
    vids = [n for n in up if n.lower().endswith(EXT_V)]
    assert vids, "Aucune vidéo reconnue (formats : mp4, mov, m4v, avi, mkv, webm)."
    VIDEO = f"/content/montage/entree/{vids[0]}"
    shutil.move(vids[0], VIDEO)
MUSIQUE = None
if ajouter_une_musique:
    print("👉 Sélectionnez la musique (mp3, wav, m4a…) :")
    up = files.upload()
    if up:
        n = list(up)[0]; MUSIQUE = f"/content/montage/entree/{n}"; shutil.move(n, MUSIQUE)
if logo and autre_logo:
    print("👉 Sélectionnez votre logo (PNG, fond transparent de préférence) :")
    up = files.upload()
    if up:
        n = list(up)[0]; R.fichier_logo = f"/content/montage/entree/{n}"; shutil.move(n, R.fichier_logo)
DUREE = M.duree_video(VIDEO)
print(f"✅ Vidéo : {os.path.basename(VIDEO)} ({DUREE:.1f} s)" + (f" — musique : {os.path.basename(MUSIQUE)}" if MUSIQUE else ""))

In [ ]:
#@title 4️⃣ Transcription automatique
mots_bruts = M.transcrire(VIDEO, modele=modele, langue="fr", vocabulaire=vocabulaire)
print("\n📝 Transcription :\n")
print(M.texte_transcription(mots_bruts))

In [ ]:
#@title 5️⃣ Corrections (facultatif) et plan de montage
#@markdown Format : `mot_mal_reconnu=bon_mot; autre=correction` — ex. `Locapé=Locapay; démarcheur=démarcheurs`
corrections = "Locapé=Locapay; Lokapay=Locapay; Locapaï=Locapay" #@param {type:"string"}
mots = M.appliquer_corrections(mots_bruts, corrections)
PLAN = M.construire_plan(mots, DUREE, R)
print(M.resume_plan(PLAN))
print("\nLes mots en MAJUSCULES seront en gras. Relancez cette étape après une correction.")

In [ ]:
#@title 6️⃣ Montage et téléchargement
from IPython.display import Video, display
os.makedirs("/content/montage/sortie", exist_ok=True)
base = os.path.splitext(os.path.basename(VIDEO))[0]
SORTIE = f"/content/montage/sortie/{base}_monte.mp4"
print("🎬 Montage en cours…")
M.monter(VIDEO, PLAN, R, SORTIE, musique=MUSIQUE)
M.ecrire_srt(PLAN, SORTIE.replace(".mp4", ".srt"))
open(SORTIE.replace(".mp4", "_transcription.txt"), "w", encoding="utf-8").write(M.texte_transcription(mots))
print("✅ Terminé !")
display(Video(SORTIE, embed=True, width=320))
files.download(SORTIE)

---
**Fichiers produits** (dossier `/content/montage/sortie`, panneau 📁 à gauche) : la vidéo montée, les sous-titres `.srt` et la transcription `.txt`.

**Astuces**
- Filmez en vertical, la personne au centre, avec un son propre : la transcription n'en sera que meilleure.
- Trop de coupes ? Montez « pause_max_conservee » (étape 2). Pas assez ? Baissez-la.
- Sous-titres qui cachent les mains ou le visage ? Ajustez « position_sous_titres ».